In [1]:
import sys
sys.path.insert(0, '/Users/pushpita/Documents/ML_Projects/echoagent')
from pathlib import Path
import json
from typing import Dict, List, Any
from backend.agents.graph_state import PlaylistGraphState
from backend.agents.prompt_parser import parse_prompt
from backend.agents.prompt_parser import LLMClient
from backend.agents.playlist_builder import build_playlist_node
from backend.agents.reranker import rerank_candidates
from backend.agents.candidate_fuser import fuse_candidates
from backend.agents.vibe_intent import VibeIntent
from backend.agents.planner_agent import PlaylistPlan
from backend.agents.planner_agent import PlannerAgent
from backend.agents.vector_retrieval_agent import retrieve_vector_candidates

/Users/pushpita/anaconda3/envs/ml_pde/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
E0000 00:00:1779462983.080346 12879372 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779462983.081370 12879372 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1779462983.081373 12879372 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779462983.081376 12879372 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1779462983.081377 1287937

In [2]:
# Database configuration
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "backend"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
DATA_DIR = PROJECT_ROOT / "database"
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

In [3]:
prompt = "Late night chill"
model = "Qwen/Qwen2.5-7B-Instruct"
llm_client = LLMClient(model_name=model, device="mps",
                    api_key=None, endpoint= None, temperature=0.7, max_new_tokens=512)
parsing = parse_prompt(prompt, prompt_schema_path=f"{SRC_DIR}/agents/prompt_schema.json", llm_client=llm_client)

database_url = f"{DATA_DIR}/music_relational.db" #get_database_url()
chroma_persist_directory = f"{DATA_DIR}/chroma_db"  # Uses default at backend/data/chroma_db

print(f"Relational Database: {database_url}")
print(f"Vector Database: {chroma_persist_directory}")

Loading checkpoint shards: 100%|██████████| 4/4 [00:47<00:00, 11.99s/it]


KeyboardInterrupt: 

In [ ]:

# Fetch relational candidates
def get_relational_candidates(limit: int = 100, database_url: str = None) -> List[Dict[str, Any]]:
    """Fetch real candidates from the relational database."""
    filters = {}  # Empty filters to get all tracks
    candidates = query_tracks(filters=filters, limit=limit, database_url=database_url)
    return candidates

# Fetch vector candidates
def get_vector_candidates(limit: int = 100, persist_directory: str = None) -> List[Dict[str, Any]]:
    """Fetch candidates from the vector database based on semantic query."""
    try:
        # Create a mock intent with a generic semantic query
        intent = type('VibeIntent', (), {
            'semantic_query': 'upbeat energetic music',
            'soft_preferences': {},
            'exclusions': {},
            'normalize': lambda: None,
            'validate': lambda: None,
        })()
        
        candidates = retrieve_vector_candidates(
            intent=intent,
            limit=limit,
            persist_directory=persist_directory,
        )
        return candidates
    except Exception as e:
        print(f"  Warning: Vector DB retrieval failed - {str(e)}")
        return []

# Fetch both relational and vector candidates
relational_candidates = get_relational_candidates(limit=100, database_url=database_url)
vector_candidates = get_vector_candidates(limit=100, persist_directory=chroma_persist_directory)

print(f"\n✓ Relational candidates: {len(relational_candidates)}")
print(f"✓ Vector candidates: {len(vector_candidates)}")

if relational_candidates:
    print(f"\nRelational sample: {relational_candidates[0].get('title', 'N/A')}")
if vector_candidates:
    print(f"Vector sample: {vector_candidates[0].get('metadata', {}).get('title', 'N/A')}")

In [ ]:
# Test 1: Fuse relational and vector candidates
if relational_candidates or vector_candidates:
    print("=" * 60)
    print("TEST 1: Candidate Fusion (Relational + Vector)")
    print("=" * 60)

    # Fuse candidates from both sources
    fused_candidates = fuse_candidates(
        relational_candidates=relational_candidates,
        vector_candidates=vector_candidates,
        relational_weight=1.0,
        vector_weight=1.0,
    )

    print(f"✓ Fused {len(fused_candidates)} candidates from both sources")
    print(f"\nFusion breakdown:")
    both = sum(1 for c in fused_candidates if len(c.get('sources', [])) > 1)
    relational_only = sum(1 for c in fused_candidates if c.get('sources') == ['relational'])
    vector_only = sum(1 for c in fused_candidates if c.get('sources') == ['vector'])
    print(f"  Both sources: {both}")
    print(f"  Relational only: {relational_only}")
    print(f"  Vector only: {vector_only}")

    if fused_candidates:
        print(f"\nTop 3 fused candidates:")
        for i, candidate in enumerate(fused_candidates[:3], 1):
            metadata = candidate.get('metadata', {})
            if isinstance(metadata, dict):
                title = metadata.get('title', 'N/A')
            else:
                title = str(metadata)[:50]
            sources = ', '.join(candidate.get('sources', []))
            score = candidate.get('retrieval_score', 0)
            print(f"  {i}. [{sources:15s}] {title} (score: {score:.2f})")
    print("✓ TEST 1 PASSED\n")
else:
    print("⚠️  Skipping TEST 1: No candidates from either source")
    fused_candidates = []

In [ ]:
# Test 2: Rerank fused candidates
if fused_candidates:
    print("=" * 60)
    print("TEST 2: Reranking Fused Candidates")
    print("=" * 60)

    intent = type('VibeIntent', (), {
        'soft_preferences': {},
        'exclusions': {},
    })()

    playlist_plan = {
        'playlist_size': 20,
        'semantic_weight': 0.25,
        'soft_preference_weight': 0.4,
        'exclusion_penalty': 2.0,
    }

    ranked = rerank_candidates(fused_candidates, intent, playlist_plan)
    print(f"✓ Reranked {len(ranked)} candidates")
    print(f"  Score range: {ranked[-1].get('score', 0):.2f} → {ranked[0].get('score', 0):.2f}")

    print(f"\nTop 5 reranked candidates:")
    for i, candidate in enumerate(ranked[:5], 1):
        metadata = candidate.get('metadata', {})
        if isinstance(metadata, dict):
            title = metadata.get('title', 'N/A')
        else:
            title = str(metadata)[:50]
        score = candidate.get('score', 0)
        print(f"  {i}. {title} (score: {score:.2f})")
    print("✓ TEST 2 PASSED\n")
else:
    print("⚠️  Skipping TEST 2: No fused candidates available")
    ranked = []

In [ ]:
# Test 3: Build playlist from reranked candidates
if ranked and len(ranked) >= 20:
    print("=" * 60)
    print("TEST 3: Build Playlist (20 tracks)")
    print("=" * 60)

    state = {
        "ranked_candidates": ranked,
        "playlist_plan": type('PlaylistPlan', (), {'playlist_size': 20})(),
    }

    result = build_playlist_node(state)
    playlist = result.get("playlist", [])

    print(f"✓ Generated playlist with {len(playlist)} tracks")
    print(f"\nFinal Playlist:")
    for i, track in enumerate(playlist, 1):
        metadata = track.get('metadata', {})
        if isinstance(metadata, dict):
            title = metadata.get('title', 'N/A')
            artist = metadata.get('artist_name', 'N/A')
        else:
            title = str(metadata)[:40]
            artist = 'N/A'
        score = track.get('score', 0)
        sources = ', '.join(track.get('sources', ['unknown'])) if 'sources' in track else 'N/A'
        print(f"  {i:2d}. [{sources:15s}] {title[:40]:40s} - {artist[:20]:20s} ({score:6.2f})")
    print("✓ TEST 3 PASSED\n")
elif ranked:
    print(f"⚠️  Skipping TEST 3: Only {len(ranked)} ranked candidates (need 20)")
else:
    print("⚠️  Skipping TEST 3: No ranked candidates available")

In [ ]:
# Summary
print("=" * 60)
print("✓ PIPELINE TESTS COMPLETED")
print("=" * 60)
print(f"""
Full pipeline tested:
  ✓ Relational database retrieval: {len(relational_candidates)} candidates
  ✓ Vector database retrieval: {len(vector_candidates)} candidates
  ✓ Candidate fusion: {len(fused_candidates)} fused
  ✓ Reranking: {len(ranked)} ranked
  ✓ Playlist building: {'Success' if len(ranked) >= 20 else 'Insufficient data'}

Pipeline tested up to: build_playlist_node()

Tests executed:
  1. Candidate Fusion (Relational + Vector)
  2. Reranking Fused Candidates
  3. Build Playlist (20 tracks)

Next steps would test: critique_node() and full graph execution
""")